# Lab 5 — DSPy: Otimização Automática de Pipeline RAG
## Compilação com BootstrapFewShot e Comparativo Antes/Depois

**Aula 11 · MBA RAG & CAG Aplicados a Direito e Segurança Pública**

**Objetivo:** Definir um pipeline RAG com módulos DSPy (Retrieve + ChainOfThought), compilar com BootstrapFewShot usando Faithfulness como métrica, e comparar os prompts e respostas antes e depois da otimização.

**Duração estimada:** 60 minutos

---

### Fluxo DSPy

```
DESENVOLVEDOR DEFINE:
  1. Módulos: Retrieve(k=5) + ChainOfThought
  2. Métrica: Faithfulness + Answer Accuracy
  3. Dataset: 50 pares Q&A jurídicos
         |
         ▼
  BootstrapFewShot Compiler
  • Gera demonstrações raciocínio
  • Seleciona os melhores exemplos
  • Otimiza estrutura do prompt
         |
         ▼
PROGRAMA COMPILADO:
  Prompt otimizado com:
  • Instruções precisas
  • 3-5 exemplos few-shot selecionados
  • Formato de resposta padronizado
```

## Etapa 0 — Instalação

In [ ]:
!pip install -q dspy-ai==2.4.17
!pip install -q openai==1.30.0  # LLM backend

print("✅ DSPy instalado!")

## Etapa 1 — Configuração do LLM

Substitua `SUA_CHAVE_AQUI` pela sua chave da OpenAI. Para uso com Ollama (gratuito), veja o comentário alternativo.

In [ ]:
import dspy
import os

# =============================================================
# OPÇÃO 1: OpenAI GPT-4o-mini (recomendado para lab)
# =============================================================
# os.environ["OPENAI_API_KEY"] = "SUA_CHAVE_AQUI"
# lm = dspy.LM("openai/gpt-4o-mini", max_tokens=500)

# =============================================================
# OPÇÃO 2: Ollama local (gratuito, requer Ollama instalado)
# =============================================================
# lm = dspy.LM("ollama_chat/llama3.2", api_base="http://localhost:11434")

# =============================================================
# OPÇÃO 3: Modo simulação (para entender o código sem API)
# =============================================================
# Usamos um mock LM para demonstração didática
# Em produção, substitua por uma das opções acima

class MockLM:
    """
    LM mock para demonstração didática sem necessidade de API.
    Retorna respostas pré-definidas para as queries do lab.
    """
    
    RESPOSTAS = {
        "tráfico privilegiado": {
            "rationale": """O Art. 33, §4º da Lei 11.343/2006 estabelece a minorante do tráfico 
            privilegiado. Para aplicá-la, devem estar presentes cumulativamente: (1) primariedade 
            do agente — sem condenação anterior transitada em julgado; (2) bons antecedentes — 
            ausência de outros registros criminais; (3) não dedicação a atividades criminosas — 
            o tráfico não pode ser ocupação habitual; (4) não integração em organização criminosa. 
            A redução é de 1/6 a 2/3, conforme o caso concreto.""",
            "answer": "Os requisitos são cumulativos: primariedade, bons antecedentes, não "
                      "dedicação a atividades criminosas e não integração em organização criminosa. "
                      "Redução de pena: 1/6 a 2/3 (Art. 33, §4º, Lei 11.343/2006)."
        },
        "prisão preventiva": {
            "rationale": """O Art. 312 do CPP prevê os fundamentos da prisão preventiva. 
            São quatro hipóteses: garantia da ordem pública (evitar reiteração criminosa), 
            garantia da ordem econômica (crimes contra o sistema financeiro), conveniência 
            da instrução criminal (proteger provas e testemunhas), e assegurar a aplicação 
            da lei penal (evitar fuga). Além dos fundamentos, é necessária a presença 
            cumulativa de: prova da existência do crime e indício suficiente de autoria.""",
            "answer": "A prisão preventiva (Art. 312 CPP) exige: prova da existência do crime, "
                      "indício de autoria, e um dos fundamentos: garantia da ordem pública, "
                      "econômica, instrução criminal ou aplicação da lei penal."
        },
        "habeas corpus": {
            "rationale": """O habeas corpus é garantia constitucional prevista no Art. 5º, LXVIII 
            da CF/88 para proteger a liberdade de locomoção. Pode ser preventivo (salvo-conduto, 
            para ameaça futura) ou liberatório (para coação presente). O STF e STJ têm 
            jurisprudência consolidada sobre seu uso: não cabe para discutir provas, não 
            substitui recurso previsto, mas é cabível para ilegalidade flagrante.""",
            "answer": "Habeas corpus (Art. 5º, LXVIII CF) protege a liberdade de locomoção. "
                      "Preventivo: salvo-conduto. Liberatório: relaxamento da prisão ilegal. "
                      "Impetrado no STJ contra ato de TJ; no STF contra ato do STJ."
        }
    }
    
    def __call__(self, *args, **kwargs):
        # Retorna formato compatível com DSPy
        prompt = str(args[0] if args else kwargs.get("prompt", "")).lower()
        for chave, resposta in self.RESPOSTAS.items():
            if chave in prompt:
                return [{"text": f"Raciocínio: {resposta['rationale']}\nResposta: {resposta['answer']}"}]
        return [{"text": "Com base no contexto fornecido, a resposta mais precisa seria obtida analisando os artigos mencionados e a jurisprudência aplicável ao caso."}]


# Para lab sem API - usar simulação
# Em produção: descomentar uma das OPÇÕES acima
print("⚠️  Modo simulação ativo (MockLM)")
print("   Para usar LLM real, descomentar OPÇÃO 1 ou OPÇÃO 2 acima")
print("")
print("✅ Ambiente DSPy configurado")

## Etapa 2 — Retriever Local

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List

# Corpus jurídico para o retriever
CORPUS_JURIDICO = [
    "Art. 33, §4º Lei 11.343/2006: Tráfico privilegiado. Agente primário, bons antecedentes, não dedicado a atividades criminosas, não integrante de organização criminosa. Redução de 1/6 a 2/3.",
    "Art. 312 CPP: Prisão preventiva. Fundamentos: garantia da ordem pública, ordem econômica, conveniência da instrução criminal, assegurar aplicação da lei penal. Requisitos: prova do crime e indício de autoria.",
    "Art. 5º, LXVIII CF/88: Habeas corpus. Conceder-se-á habeas corpus sempre que alguém sofrer ou se achar ameaçado de sofrer violência ou coação em sua liberdade de locomoção, por ilegalidade ou abuso de poder.",
    "Art. 157 CP: Roubo. Subtrair coisa móvel alheia mediante grave ameaça ou violência. Pena: 4 a 10 anos. Majorado com arma de fogo: de 2/3 até o dobro.",
    "Lei 12.850/2013: Organização criminosa. Associação de 4 ou mais pessoas estruturalmente ordenada. Pena: 3 a 8 anos. Benefícios para colaboradores: redução de 1/3 a 2/3.",
    "Art. 33 Lei 11.343/2006: Tráfico de drogas. Importar, exportar, vender, guardar drogas sem autorização. Pena: 5 a 15 anos e 500 a 1.500 dias-multa. Crime inafiançável.",
    "Art. 240 CPP: Busca e apreensão. Domiciliar: mandado judicial. Pessoal: prescinde de mandado em flagrante ou fundada suspeita. Inviolabilidade domiciliar excepcionada por flagrante, desastre, socorro.",
    "Lei 8.072/90: Crimes hediondos. Progressão de regime: 40% primário, 60% reincidente. Vedados: fiança, graça, anistia. Inclui latrocínio, extorsão com morte, estupro, tráfico.",
    "Art. 158-A CPP: Cadeia de custódia. Preservação dos vestígios desde o reconhecimento até o descarte. Fases: reconhecimento, isolamento, fixação, coleta, acondicionamento, transporte, recebimento, processamento, armazenamento.",
    "Art. 4º Lei 12.850/2013: Colaboração premiada. Benefícios: perdão judicial, redução de pena, progressão especial. Eficácia probatória: corroborada por outros elementos. Homologação pelo juiz.",
]

# Carregar modelo de embedding
print("📥 Carregando modelo para retriever...")
modelo_retriever = SentenceTransformer("BAAI/bge-m3")
embeddings_corpus = modelo_retriever.encode(CORPUS_JURIDICO, normalize_embeddings=True)
print("✅ Retriever pronto")


def retriever_local(query: str, k: int = 3) -> List[str]:
    """
    Retriever semântico local que retorna os k documentos mais relevantes.
    Em produção: substituir por cliente OpenSearch.
    """
    q_emb = modelo_retriever.encode(query, normalize_embeddings=True)
    scores = embeddings_corpus @ q_emb
    indices = np.argsort(scores)[::-1][:k]
    return [CORPUS_JURIDICO[i] for i in indices]


# Teste do retriever
docs = retriever_local("tráfico privilegiado requisitos", k=2)
print("\nTeste do retriever:")
for i, doc in enumerate(docs, 1):
    print(f"  [{i}] {doc[:80]}...")

## Etapa 3 — Definição do Pipeline RAG Manual (Sem DSPy)

In [ ]:
# Pipeline RAG manual - como seria sem DSPy
# Este é o "antes" da otimização

PROMPT_MANUAL_TEMPLATE = """
Responda à pergunta jurídica com base no contexto fornecido.

Contexto:
{contexto}

Pergunta: {pergunta}

Resposta:"""


class PipelineRAGManual:
    """Pipeline RAG com prompt escrito manualmente (baseline)."""
    
    def __init__(self, retriever, lm):
        self.retriever = retriever
        self.lm = lm
    
    def responder(self, pergunta: str) -> dict:
        # 1. Recuperar contexto
        docs = self.retriever(pergunta, k=3)
        contexto = "\n\n".join(docs)
        
        # 2. Construir prompt manual
        prompt = PROMPT_MANUAL_TEMPLATE.format(
            contexto=contexto,
            pergunta=pergunta
        )
        
        # 3. Chamar LLM
        resposta = self.lm(prompt)
        texto_resposta = resposta[0]["text"] if isinstance(resposta, list) else str(resposta)
        
        return {
            "pergunta": pergunta,
            "contexto": contexto,
            "prompt": prompt,
            "resposta": texto_resposta
        }


mock_lm = MockLM()
pipeline_manual = PipelineRAGManual(retriever_local, mock_lm)

# Teste do pipeline manual
resultado_manual = pipeline_manual.responder(
    "Quais são os requisitos para o tráfico privilegiado?"
)

print("PIPELINE MANUAL (SEM DSPy)")
print("=" * 60)
print(f"\nPERGUNTA: {resultado_manual['pergunta']}")
print(f"\nPROMPT enviado ao LLM:")
print("-" * 40)
print(resultado_manual['prompt'][:400] + "...")
print("-" * 40)
print(f"\nRESPOSTA: {resultado_manual['resposta'][:200]}...")

## Etapa 4 — Definição do Pipeline DSPy

In [ ]:
import dspy

# ============================================================
# ASSINATURA DSPy
# Define campos de entrada/saída sem especificar o prompt
# O DSPy compila o prompt automaticamente!
# ============================================================

class AssistenciaJuridica(dspy.Signature):
    """
    Assinatura para assistência jurídica baseada em contexto.
    O DSPy usa a docstring e os campos para gerar o prompt.
    """
    
    # Campos de entrada
    contexto = dspy.InputField(
        desc="Trechos relevantes de legislação, jurisprudência e doutrina"
    )
    pergunta = dspy.InputField(
        desc="Questão jurídica a ser respondida com base no contexto"
    )
    
    # Campos de saída
    raciocinio = dspy.OutputField(
        desc="Análise jurídica passo a passo: fundamento legal, aplicação ao caso, conclusão"
    )
    resposta = dspy.OutputField(
        desc="Resposta objetiva e fundamentada com artigo legal e pena aplicável"
    )


# ============================================================
# MÓDULO DSPy — Pipeline RAG com ChainOfThought
# ============================================================

class PipelineRAGDSPy(dspy.Module):
    """
    Pipeline RAG implementado como módulo DSPy.
    Combina Retrieve + ChainOfThought em um programa compilável.
    """
    
    def __init__(self, retriever, k: int = 3):
        super().__init__()
        self.retriever = retriever
        self.k = k
        # ChainOfThought aplica raciocínio step-by-step antes de responder
        self.chain_of_thought = dspy.ChainOfThought(AssistenciaJuridica)
    
    def forward(self, pergunta: str) -> dspy.Prediction:
        """
        Executa o pipeline: recupera contexto → raciocina → responde.
        O método forward() é chamado automaticamente pelo DSPy.
        """
        # Etapa 1: Recuperar documentos relevantes
        docs_recuperados = self.retriever(pergunta, k=self.k)
        contexto = "\n".join([f"[{i+1}] {doc}" for i, doc in enumerate(docs_recuperados)])
        
        # Etapa 2: Raciocinar e responder com CoT
        resultado = self.chain_of_thought(
            contexto=contexto,
            pergunta=pergunta
        )
        
        return resultado


# Para usar DSPy com LLM real:
# dspy.configure(lm=lm)
# pipeline_dspy = PipelineRAGDSPy(retriever_local)

print("✅ Módulo DSPy definido!")
print("\nAssinatura compilada:")
print(f"  Entradas: contexto, pergunta")
print(f"  Saídas:   raciocinio, resposta")
print(f"  Módulo:   ChainOfThought (raciocínio step-by-step)")

## Etapa 5 — Dataset e Métrica para Compilação

In [ ]:
# Dataset de treinamento: pares pergunta-resposta gold
# Em produção: usar dataset de avaliação real com especialistas

dataset_juridico = [
    dspy.Example(
        pergunta="Quais são os requisitos do tráfico privilegiado?",
        resposta_gold="Requisitos cumulativos: primariedade, bons antecedentes, não dedicação a atividades criminosas e não integração em organização criminosa (Art. 33, §4º, Lei 11.343/2006). Redução de 1/6 a 2/3."
    ).with_inputs("pergunta"),
    dspy.Example(
        pergunta="Quando cabe prisão preventiva?",
        resposta_gold="A prisão preventiva cabe quando presentes: prova do crime, indício de autoria, e fundamento (ordem pública, ordem econômica, instrução criminal ou aplicação da lei penal). Art. 312 do CPP."
    ).with_inputs("pergunta"),
    dspy.Example(
        pergunta="O que é habeas corpus e quando cabe?",
        resposta_gold="Habeas corpus (Art. 5º, LXVIII CF) é remédio constitucional para proteger a liberdade de locomoção ameaçada por ilegalidade ou abuso de poder. Cabe preventivo (salvo-conduto) ou liberatório (relaxamento)."
    ).with_inputs("pergunta"),
    dspy.Example(
        pergunta="Qual a pena para tráfico de drogas?",
        resposta_gold="Pena para tráfico (Art. 33, Lei 11.343/2006): reclusão de 5 a 15 anos e 500 a 1.500 dias-multa. Crime inafiançável. Com privilégio (§4º): redução de 1/6 a 2/3."
    ).with_inputs("pergunta"),
    dspy.Example(
        pergunta="O que diferencia roubo de extorsão?",
        resposta_gold="No roubo (Art. 157 CP), o agente subtrai o bem diretamente com violência/ameaça. Na extorsão (Art. 158 CP), a vantagem econômica depende de comportamento da vítima (fazer, tolerar ou omitir algo)."
    ).with_inputs("pergunta"),
    dspy.Example(
        pergunta="Como funciona a colaboração premiada?",
        resposta_gold="Colaboração premiada (Art. 4º, Lei 12.850/2013): acordo entre MP e colaborador que fornece informações eficazes sobre organização criminosa. Benefícios: perdão judicial, redução de 1/3 a 2/3, progressão especial. Eficácia: exige corroboração."
    ).with_inputs("pergunta"),
]

# Dividir em treino (70%) e validação (30%)
n_treino = int(len(dataset_juridico) * 0.7)
treino = dataset_juridico[:n_treino]
validacao = dataset_juridico[n_treino:]

print(f"📊 Dataset jurídico:")
print(f"  Total: {len(dataset_juridico)} exemplos")
print(f"  Treino: {len(treino)} exemplos")
print(f"  Validação: {len(validacao)} exemplos")


# Métrica de avaliação: Faithfulness por sobreposição de termos-chave
def metrica_faithfulness(exemplo, predicao, trace=None) -> float:
    """
    Métrica simples de faithfulness: sobreposição de termos-chave.
    
    Em produção: usar RAGAS com LLM-as-Judge para avaliação semântica:
    from ragas.metrics import faithfulness, answer_relevancy
    """
    resposta_gold = exemplo.resposta_gold.lower()
    resposta_pred = getattr(predicao, 'resposta', str(predicao)).lower()
    
    # Extrair termos jurídicos importantes do gold
    termos = [t for t in resposta_gold.split() if len(t) > 4]
    
    # Calcular sobreposição
    cobertura = sum(1 for t in termos if t in resposta_pred) / max(len(termos), 1)
    return cobertura


print("\n✅ Dataset e métrica configurados!")

## Etapa 6 — Compilação com BootstrapFewShot

In [ ]:
# Demonstração completa do processo de compilação DSPy
# NOTA: Com LLM real, this compiles and optimizes automatically
# Aqui mostramos o processo conceptual passo a passo

print("=" * 65)
print("PROCESSO DE COMPILAÇÃO DSPy — BootstrapFewShot")
print("=" * 65)

print("""
Com LLM real, o código de compilação seria:

  from dspy.teleprompt import BootstrapFewShot
  
  # 1. Definir o compilador
  compilador = BootstrapFewShot(
      metric=metrica_faithfulness,  # Métrica de avaliação
      max_bootstrapped_demos=4,      # Máximo de exemplos few-shot
      max_labeled_demos=4,           # Exemplos do dataset de treino
  )
  
  # 2. Compilar (otimizar) o pipeline
  pipeline_otimizado = compilador.compile(
      student=PipelineRAGDSPy(retriever_local),  # Módulo a otimizar
      teacher=PipelineRAGDSPy(retriever_local),  # Teacher para bootstrap
      trainset=treino,                            # Dataset de treino
  )
  
  # 3. Avaliar antes e depois
  from dspy.evaluate import Evaluate
  avaliador = Evaluate(devset=validacao, metric=metrica_faithfulness)
  score_antes = avaliador(PipelineRAGDSPy(retriever_local))
  score_depois = avaliador(pipeline_otimizado)
""")

# Simulação do resultado da compilação
print("\n" + "=" * 65)
print("RESULTADO SIMULADO DA COMPILAÇÃO")
print("=" * 65)
print(f"Score antes (pipeline manual):   0.62")
print(f"Score depois (DSPy otimizado):   0.81")
print(f"Melhoria:                        +30.6%")

In [ ]:
# Demonstração visual: Prompt ANTES vs DEPOIS da compilação
# Mostra como o DSPy transforma um prompt manual em um prompt otimizado

PROMPT_ANTES = """
Responda à pergunta jurídica com base no contexto fornecido.

Contexto:
{contexto}

Pergunta: {pergunta}

Resposta:"""

PROMPT_DEPOIS = """
Você é um assistente jurídico especializado em Direito Penal e Segurança
Pública brasileiro. Analise o contexto jurídico e responda com precisão,
identificando o fundamento legal específico.

---
EXEMPLO 1:
Contexto: [1] Art. 33, §4º Lei 11.343/2006: Tráfico privilegiado. Agente primário, bons antecedentes, não dedicado a atividades criminosas, não integrante de organização criminosa. Redução de 1/6 a 2/3.
Pergunta: Quais os requisitos do tráfico privilegiado?
Raciocínio: O Art. 33, §4º estabelece quatro requisitos cumulativos: (1) primariedade — sem condenação anterior; (2) bons antecedentes — sem registros criminais; (3) não dedicação a atividades criminosas — o tráfico não é ocupação habitual; (4) não integração em organização criminosa. Todos devem estar presentes simultaneamente.
Resposta: Requisitos cumulativos: primariedade, bons antecedentes, não dedicação a atividades criminosas, não integração em organização criminosa (Art. 33, §4º, Lei 11.343/2006). Redução: 1/6 a 2/3.
---
EXEMPLO 2:
Contexto: [1] Art. 312 CPP: Prisão preventiva. Fundamentos: garantia da ordem pública, ordem econômica, conveniência da instrução criminal, assegurar aplicação da lei penal. Requisitos: prova do crime e indício de autoria.
Pergunta: Quando cabe prisão preventiva?
Raciocínio: O Art. 312 CPP exige dois elementos objetivos: prova da existência do crime e indício suficiente de autoria. Além disso, exige um dos quatro fundamentos: garantia da ordem pública (evitar reiteração), ordem econômica (crimes financeiros), instrução criminal (proteger provas) ou aplicação da lei penal (evitar fuga).
Resposta: Cabe quando presentes: prova do crime, indício de autoria, e fundamento do Art. 312 CPP (ordem pública, econômica, instrução ou aplicação da pena).
---

Contexto: {contexto}
Pergunta: {pergunta}
Raciocínio:
Resposta:"""

print("=" * 65)
print("COMPARATIVO DE PROMPTS: ANTES vs DEPOIS DA COMPILAÇÃO DSPy")
print("=" * 65)

print("\n📝 ANTES (prompt manual):")
print("-" * 40)
print(PROMPT_ANTES)
print(f"\nCaracterísticas: {len(PROMPT_ANTES.split())} palavras | Sem exemplos | Sem instrução especializada")

print("\n📝 DEPOIS (DSPy BootstrapFewShot compilado):")
print("-" * 40)
print(PROMPT_DEPOIS[:600] + "...")
print(f"\nCaracterísticas: {len(PROMPT_DEPOIS.split())} palavras | 2 exemplos few-shot | Raciocínio estruturado")

print("""
Diferenças chave após compilação:
1. Instrução especializada: papel definido explicitamente
2. Exemplos few-shot: 2-4 demonstrações reais selecionadas por score
3. Campo 'Raciocínio': força CoT (Chain of Thought) antes da resposta
4. Formato estruturado: resposta sempre inclui artigo e consequência
5. Contexto numerado: facilita referência na resposta
""")

## Etapa 7 — Avaliação Comparativa

In [ ]:
# Simulação da avaliação comparativa nos exemplos de validação

# Respostas simuladas dos dois pipelines para os exemplos de validação
resultados_comparativos = [
    {
        "pergunta": "Qual a pena para tráfico de drogas?",
        "gold": "Reclusão de 5 a 15 anos e 500 a 1.500 dias-multa (Art. 33, Lei 11.343/2006). Com privilégio: redução de 1/6 a 2/3.",
        "manual": "A pena para tráfico é de reclusão. O crime é grave e pode ter agravantes.",
        "dspy": "Tráfico de drogas (Art. 33, Lei 11.343/2006): reclusão de 5 a 15 anos e 500 a 1.500 dias-multa. Com tráfico privilegiado (§4º): redução de 1/6 a 2/3 se agente primário e sem organização criminosa."
    },
    {
        "pergunta": "O que diferencia roubo de extorsão?",
        "gold": "No roubo, agente subtrai bem com violência. Na extorsão, vantagem depende do comportamento da vítima.",
        "manual": "São crimes diferentes no Código Penal, com penas diferentes.",
        "dspy": "Roubo (Art. 157 CP): subtração direta com violência ou ameaça — pena 4-10 anos. Extorsão (Art. 158 CP): vantagem econômica depende de comportamento da vítima (fazer/tolerar/omitir) — pena 4-10 anos. Diferença essencial: na extorsão, há colaboração coagida da vítima."
    },
]


def avaliar_resposta(resposta: str, gold: str) -> dict:
    """Avalia qualidade da resposta em múltiplas dimensões."""
    # Faithfulness: sobreposição de termos
    termos_gold = [t.lower() for t in gold.split() if len(t) > 4]
    resp_lower = resposta.lower()
    faith = sum(1 for t in termos_gold if t in resp_lower) / max(len(termos_gold), 1)
    
    # Completude: cobre artigos e números
    numeros_gold = [t for t in gold.split() if any(c.isdigit() for c in t)]
    completude = sum(1 for n in numeros_gold if n in resposta) / max(len(numeros_gold), 1)
    
    # Precisão técnica: menciona artigo
    tem_artigo = "art." in resposta.lower() or "§" in resposta
    
    return {
        "faithfulness": faith,
        "completude": completude,
        "tem_artigo": tem_artigo,
        "score_geral": (faith * 0.5 + completude * 0.3 + int(tem_artigo) * 0.2)
    }


print("=" * 70)
print("AVALIAÇÃO COMPARATIVA: MANUAL vs DSPy OTIMIZADO")
print("=" * 70)

scores_manual = []
scores_dspy = []

for r in resultados_comparativos:
    av_manual = avaliar_resposta(r["manual"], r["gold"])
    av_dspy = avaliar_resposta(r["dspy"], r["gold"])
    scores_manual.append(av_manual["score_geral"])
    scores_dspy.append(av_dspy["score_geral"])
    
    print(f"\nPERGUNTA: {r['pergunta']}")
    print(f"  Manual:  Score={av_manual['score_geral']:.2f} | Faith={av_manual['faithfulness']:.2f} | Artigo={'✅' if av_manual['tem_artigo'] else '❌'}")
    print(f"           \"{r['manual'][:80]}...\"")
    print(f"  DSPy:    Score={av_dspy['score_geral']:.2f} | Faith={av_dspy['faithfulness']:.2f} | Artigo={'✅' if av_dspy['tem_artigo'] else '❌'}")
    print(f"           \"{r['dspy'][:80]}...\"")

import numpy as np
print(f"\n{'='*70}")
print(f"Score médio Manual: {np.mean(scores_manual):.3f}")
print(f"Score médio DSPy:   {np.mean(scores_dspy):.3f}")
melhoria = (np.mean(scores_dspy) / np.mean(scores_manual) - 1) * 100
print(f"Melhoria DSPy:      {melhoria:+.1f}%")

## Exercício Prático

> **Exercício:** Adicione 4 novos exemplos ao `dataset_juridico` cobrindo temas de legislação de segurança pública (ex: Estatuto do Desarmamento, Lei de Organizações Criminosas). Modifique a `AssistenciaJuridica` para adicionar um campo de saída `fundamento_legal` que sempre cite o artigo específico. Avalie se esse campo adicional melhora a qualidade das respostas.

## Pergunta de Reflexão

> **Reflexão:** O DSPy otimiza prompts baseado em métricas definidas pelo desenvolvedor. Se a métrica fosse apenas "resposta curta" em vez de "faithfulness", que comportamento o compilador induziria? Como isso impacta a responsabilidade (accountability) em sistemas de assessoria jurídica automatizada?

---

**Próximo:** Lab 6 — Dashboard Comparativo LangFuse